In [26]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import HuberRegressor, ElasticNet
from catboost import CatBoostRegressor, Pool, cv
from xgboost import XGBRegressor

import optuna
from optuna.samplers import TPESampler

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [3]:
# read data

train_data_raw = pd.read_csv('.data/train.csv')
test_data_raw = pd.read_csv('.data/test.csv')

In [4]:
# EDA: train data first look

# print(train_data_raw.head(10))
# train_data_raw.info()
# train_data_raw.describe()

In [5]:
# EDA: test data first look

# test_data_raw.head(10)
# test_data_raw.info()
# test_data_raw.describe()

In [6]:
# data cleaning
###################


# drop id column
train_data = train_data_raw.drop(columns=['Id'])
test_data = test_data_raw.drop(columns=['Id'])

# there is couple columns that have the numerical dtype but actually categorical, so we need to convert them to str or object dtype

columns_to_convert_n2s = ["MSSubClass"]

for col in columns_to_convert_n2s:
    train_data[col] = train_data[col].astype(str)
    test_data[col] = test_data[col].astype(str)

# seperate numerical and categorical columns, and target column
target_col = "SalePrice"

# all columns except target column are features
feature_cols = [c for c in train_data.columns if c != target_col]

numerical_cols = train_data[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = train_data[feature_cols].select_dtypes(include=["object", "string"]).columns.tolist()

# handle nan, none missing values, etc.
# for almost all columns, nan means that feature is not present, so it can be filled with 0 for numerical columns, and "None" for categorical columns

for col in numerical_cols:
    train_data[col] = train_data[col].fillna(0)
    test_data[col] = test_data[col].fillna(0)

for col in categorical_cols:
    train_data[col] = train_data[col].fillna("None")
    test_data[col] = test_data[col].fillna("None")

In [7]:
# double check if there is any missing value left

print(train_data.isnull().sum().sum())
print(test_data.isnull().sum().sum())

0
0


# some notes on data

- year built and year remod needs special scaling
- for some columns NA means lack of the said feature
- datetime conversion from MoSold: Month Sold (MM), YrSold: Year Sold (YYYY)

In [8]:
train_data[["YrSold", "GarageYrBlt", "YearBuilt", "YearRemodAdd"]].describe()

,YrSold,GarageYrBlt,YearBuilt,YearRemodAdd
count,1460.000000,1460.000000,1460.000000,1460.000000
mean,2007.815753,1868.739726,1971.267808,1984.865753
std,1.328095,453.697295,30.202904,20.645407
min,2006.000000,0.000000,1872.000000,1950.000000
25%,2007.000000,1958.000000,1954.000000,1967.000000
50%,2008.000000,1977.000000,1973.000000,1994.000000
75%,2009.000000,2001.000000,2000.000000,2004.000000
max,2010.000000,2010.000000,2010.000000,2010.000000


In [9]:
# Feature engineering
#####################

# addition of date based columns

train_data["AgeAtSale"] = train_data["YrSold"] - train_data["YearBuilt"]
train_data["TimeSinceRemodel"] = train_data["YrSold"] - train_data["YearRemodAdd"]

# garage age: should be 0 if there is no garage, and should be age of garage if there is a garage

train_data["GarageAge"] = (train_data["YrSold"] - train_data["GarageYrBlt"]).where(train_data["GarageType"] != "None", 0)


# same feature engineering for test data
test_data["AgeAtSale"] = test_data["YrSold"] - test_data["YearBuilt"]
test_data["TimeSinceRemodel"] = test_data["YrSold"] - test_data["YearRemodAdd"]
test_data["GarageAge"] = (test_data["YrSold"] - test_data["GarageYrBlt"]).where(test_data["GarageType"] != "None", 0)

In [10]:
# recalculate numerical and categorical columns after feature engineering

feature_cols = [c for c in train_data.columns if c != target_col]
numerical_cols = train_data[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = train_data[feature_cols].select_dtypes(include=["object", "string"]).columns.tolist()


In [11]:
# data preprocessing
#####################

# Step 1: define transformers for numerical and categorical columns

onehot_encoder = OneHotEncoder(drop="if_binary", handle_unknown="warn")
scaler = StandardScaler()


In [12]:
# Step 2: preprocessing of numerical features

# Branch 1: preprocessing of numerical features for scale-sensitive (linear) models (GaussianProcessRegressor, PLSRegression, HuberRegressor, ElasticNet)
transformer_1 = ColumnTransformer([
    ("num", scaler, numerical_cols),
    ("cat", onehot_encoder, categorical_cols)
])

# Branch 2: preprocessing of numerical features for the rest of the models, except CatBoost

transformer_2 = ColumnTransformer([
    ("num", "passthrough", numerical_cols),
    ("cat", onehot_encoder, categorical_cols)
])

# Step 3: preprocessing of of target variable, log transformation is applied to make the distribution more normal, which is beneficial for regression models

y_train = np.log1p(train_data[target_col])

# Step 4: apply the transformations to the training and test data

X_train_transformed_1 = transformer_1.fit_transform(train_data[feature_cols])
X_train_transformed_2 = transformer_2.fit_transform(train_data[feature_cols])

X_test_transformed_1 = transformer_1.transform(test_data[feature_cols])
X_test_transformed_2 = transformer_2.transform(test_data[feature_cols])

# Step 5: split the training data into train and validation sets for model evaluation
# 0 for catboost, 1 for scale-sensitive models, 2 for the rest of the models

X_train_0, X_val_0, y_train_0, y_val_0 = train_test_split(train_data[feature_cols], y_train, test_size=0.2, random_state=42)
X_train_1, X_val_1, y_train_1, y_val_1 = train_test_split(X_train_transformed_1, y_train, test_size=0.2, random_state=42)
X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(X_train_transformed_2, y_train, test_size=0.2, random_state=42)


c:\Users\Doruk\Repos\personal\fun\kaggle_house_price_2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 6, 16, 17, 31, 32, 42] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
c:\Users\Doruk\Repos\personal\fun\kaggle_house_price_2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 6, 16, 17, 31, 32, 42] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


In [13]:
# Model training and evaluation
################################

# all training will include cross validation with stratified k-fold with 5 folds

# Branch 1: catboost

train_pool = Pool(
    data=X_train_0,
    label=y_train_0,
    cat_features=categorical_cols  # using column names from your dataframe
)

cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "learning_rate": 0.03,
    "depth": 6,
    "random_seed": 42,
    "iterations": 10000,
}

cv_results = cv(
    pool=train_pool,
    params=cat_params,
    fold_count=5,
    shuffle=True,
    partition_random_seed=42,
    early_stopping_rounds=500,
    verbose=1000
)

best_iter = int(cv_results["test-RMSE-mean"].idxmin()) + 1
best_cv_rmse = float(cv_results.loc[best_iter - 1, "test-RMSE-mean"])

print(f"Best CV iteration: {best_iter}")
print(f"Best CV RMSE(log): {best_cv_rmse:.5f}")

Training on fold [0/5]
0:	learn: 11.6692647	test: 11.7631833	best: 11.7631833 (0)	total: 216ms	remaining: 36m 3s
1000:	learn: 0.0753771	test: 0.1995223	best: 0.1994950 (997)	total: 58.9s	remaining: 8m 49s
2000:	learn: 0.0426159	test: 0.1946118	best: 0.1944481 (1812)	total: 1m 55s	remaining: 7m 42s
3000:	learn: 0.0276518	test: 0.1939495	best: 0.1938407 (2805)	total: 2m 45s	remaining: 6m 25s

bestTest = 0.1937281523
bestIteration = 3453

Training on fold [1/5]
0:	learn: 11.6918660	test: 11.6814999	best: 11.6814999 (0)	total: 40.4ms	remaining: 6m 43s
1000:	learn: 0.0750093	test: 0.1998758	best: 0.1998622 (999)	total: 45.9s	remaining: 6m 52s
2000:	learn: 0.0417487	test: 0.1950551	best: 0.1950551 (2000)	total: 1m 33s	remaining: 6m 13s
3000:	learn: 0.0273265	test: 0.1940826	best: 0.1940686 (2972)	total: 2m 23s	remaining: 5m 33s
4000:	learn: 0.0181427	test: 0.1937950	best: 0.1937465 (3709)	total: 3m 12s	remaining: 4m 49s

bestTest = 0.1937465474
bestIteration = 3709

Training on fold [2/5]
0:

In [22]:
# Model training and evaluation

# Branch 2: scale-sensitive models (GaussianProcessRegressor, PLSRegression, HuberRegressor, ElasticNet)

# GaussianProcessRegressor
kernel = ConstantKernel() * RBF() + WhiteKernel()

estimator_GP = GaussianProcessRegressor(kernel=kernel, normalize_y=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    estimator_GP,
    X_train_1.toarray(),  # convert sparse matrix to dense array for GaussianProcessRegressor
    y_train_1,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -scores
print(f"Fold RMSE(log): {rmse_scores}")
print(f"Mean RMSE(log): {rmse_scores.mean():.5f}")
print(f"Std RMSE(log): {rmse_scores.std():.5f}")

Fold RMSE(log): [0.12916086 0.13406346 0.13669435 0.10375917 0.09683276]
Mean RMSE(log): 0.12010
Std RMSE(log): 0.01650


In [24]:
# PLSRegression
estimator_PLS = PLSRegression(n_components=10)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    estimator_PLS,
    X_train_1.toarray(),  # convert sparse matrix to dense array for PLSRegression
    y_train_1,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -scores
print(f"Fold RMSE(log): {rmse_scores}")
print(f"Mean RMSE(log): {rmse_scores.mean():.5f}")
print(f"Std RMSE(log): {rmse_scores.std():.5f}")


Fold RMSE(log): [0.14332999 0.14599567 0.20407026 0.12758821 0.15402124]
Mean RMSE(log): 0.15500
Std RMSE(log): 0.02599


In [16]:
# HuberRegressor

estimator_Huber = HuberRegressor()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    estimator_Huber,
    X_train_1.toarray(),  # convert sparse matrix to dense array for HuberRegressor
    y_train_1,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -scores
print(f"Fold RMSE(log): {rmse_scores}")
print(f"Mean RMSE(log): {rmse_scores.mean():.5f}")
print(f"Std RMSE(log): {rmse_scores.std():.5f}")

Fold RMSE(log): [0.18670634 0.17498847 0.22296905 0.15092685 0.15531686]
Mean RMSE(log): 0.17818
Std RMSE(log): 0.02590


In [17]:
# ElasticNet

estimator_ElasticNet = ElasticNet(random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    estimator_ElasticNet,
    X_train_1.toarray(),  # convert sparse matrix to dense array for ElasticNet
    y_train_1,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -scores
print(f"Fold RMSE(log): {rmse_scores}")
print(f"Mean RMSE(log): {rmse_scores.mean():.5f}")
print(f"Std RMSE(log): {rmse_scores.std():.5f}")

Fold RMSE(log): [0.40608999 0.39624028 0.41064599 0.37232568 0.36505277]
Mean RMSE(log): 0.39007
Std RMSE(log): 0.01821


In [18]:
# Model training and evaluation

# Branch 3: the rest of the models; xgboost, random forest, extra trees

# RandomForestRegressor

estimator_RF = RandomForestRegressor(n_estimators=100, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    estimator_RF,
    X_train_2,
    y_train_2,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -scores
print(f"Fold RMSE(log): {rmse_scores}")
print(f"Mean RMSE(log): {rmse_scores.mean():.5f}")
print(f"Std RMSE(log): {rmse_scores.std():.5f}")

Fold RMSE(log): [0.1507532  0.1500246  0.17706057 0.13540165 0.12946977]
Mean RMSE(log): 0.14854
Std RMSE(log): 0.01647


In [19]:
# ExtraTreesRegressor

estimator_ET = ExtraTreesRegressor(n_estimators=100, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    estimator_ET,
    X_train_2,
    y_train_2,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -scores
print(f"Fold RMSE(log): {rmse_scores}")
print(f"Mean RMSE(log): {rmse_scores.mean():.5f}")
print(f"Std RMSE(log): {rmse_scores.std():.5f}")

Fold RMSE(log): [0.15408753 0.14623419 0.16339779 0.13340839 0.14094178]
Mean RMSE(log): 0.14761
Std RMSE(log): 0.01039


In [20]:
# XGBRegressor

estimator_XGB = XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    estimator_XGB,
    X_train_2,
    y_train_2,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -scores
print(f"Fold RMSE(log): {rmse_scores}")
print(f"Mean RMSE(log): {rmse_scores.mean():.5f}")
print(f"Std RMSE(log): {rmse_scores.std():.5f}")

Fold RMSE(log): [0.14920668 0.14714834 0.18345188 0.12790232 0.12269337]
Mean RMSE(log): 0.14608
Std RMSE(log): 0.02138


In [28]:
# Hyperparameter tuning with Optuna for GPRegressor

X_tr = X_train_1.toarray()
y_tr = y_train_1
X_val = X_val_1.toarray()
y_val = y_val_1

def objective_GP(trial):
    length_scale = trial.suggest_loguniform("length_scale", 0.1, 50.0)
    noise_level = trial.suggest_loguniform("noise_level", 1e-6, 1e-1)
    normalize_y = trial.suggest_categorical("normalize_y", [True, False])

    kernel = ConstantKernel(1.0) * RBF(length_scale=length_scale) + WhiteKernel(noise_level=noise_level)
    gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=normalize_y, random_state=42)

    gpr.fit(X_tr, y_tr)
    preds = gpr.predict(X_val)
    rmse = root_mean_squared_error(y_val, preds)
    return rmse

# Tune
study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study.optimize(objective_GP, n_trials=300)

print("Best RMSE (log):", study.best_value)
print("Best params:", study.best_params)

[I 2026-05-28 15:01:53,329] A new study created in memory with name: no-name-f7bea459-d584-4d1e-81d6-57366d78e794
C:\Users\Doruk\AppData\Local\Temp\ipykernel_16004\150973767.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  length_scale = trial.suggest_loguniform("length_scale", 0.1, 50.0)
C:\Users\Doruk\AppData\Local\Temp\ipykernel_16004\150973767.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  noise_level = trial.suggest_loguniform("noise_level", 1e-6, 1e-1)
[I 2026-05-28 15:01:59,724] Trial 0 finished with value: 0.13253570055792635 and parameters: {'length_scale': 1.025350969016849, 'noise_level': 0.05669849511478854, 'normalize_y': True}. Best is trial 0 wit

Best RMSE (log): 0.12449740338630581
Best params: {'length_scale': 2.94840898986105, 'noise_level': 0.025942747862994085, 'normalize_y': False}


In [31]:
# Hyperparameter tuning with Optuna for PLSRegression

X_tr = X_train_1.toarray()
y_tr = y_train_1
X_val = X_val_1.toarray()
y_val = y_val_1

def objective_PLS(trial):
    n_features = X_tr.shape[1]
    n_components = trial.suggest_int("n_components", 1, n_features)
    scale = trial.suggest_categorical("scale", [True, False])

    model = PLSRegression(n_components=n_components, scale=scale)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val).ravel()
    rmse = root_mean_squared_error(y_val, preds)
    return rmse

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study.optimize(objective_PLS, n_trials=1000)

print("Best RMSE (log):", study.best_value)
print("Best params:", study.best_params)

[I 2026-05-28 15:49:25,642] A new study created in memory with name: no-name-0987dbb4-b41a-4993-98c2-8f90f3504c37
[I 2026-05-28 15:49:25,912] Trial 0 finished with value: 0.13692969853790998 and parameters: {'n_components': 119, 'scale': True}. Best is trial 0 with value: 0.13692969853790998.
[I 2026-05-28 15:49:26,379] Trial 1 finished with value: 0.13675490875954632 and parameters: {'n_components': 190, 'scale': True}. Best is trial 1 with value: 0.13675490875954632.
[I 2026-05-28 15:49:26,416] Trial 2 finished with value: 0.1268944940593323 and parameters: {'n_components': 19, 'scale': True}. Best is trial 2 with value: 0.1268944940593323.
[I 2026-05-28 15:49:26,880] Trial 3 finished with value: 0.13793687978914007 and parameters: {'n_components': 225, 'scale': False}. Best is trial 2 with value: 0.1268944940593323.
[I 2026-05-28 15:49:27,415] Trial 4 finished with value: 5880254005782.005 and parameters: {'n_components': 264, 'scale': True}. Best is trial 2 with value: 0.1268944940

Best RMSE (log): 0.1268944940593323
Best params: {'n_components': 19, 'scale': True}


In [37]:
#Hyperparameter tuning with Optuna for XGBRegressor

X_tr = X_train_1.toarray()
y_tr = y_train_1
X_val = X_val_1.toarray()
y_val = y_val_1

def objective_XGB(trial):
    params = {
        "n_estimators": 10000,
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-3, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "gamma": trial.suggest_loguniform("gamma", 1e-8, 10.0),
        "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-8, 10.0),
        "reg_lambda": trial.suggest_loguniform("reg_lambda", 1e-8, 10.0),
        "random_state": 42,
        "n_jobs": -1,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "verbosity": 0,
    }

    model = XGBRegressor(**params)

    # use early stopping on the holdout to speed trials
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    preds = model.predict(X_val)
    rmse = root_mean_squared_error(y_val, preds)
    return rmse


study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study.optimize(objective_XGB, n_trials=250)

print("Best RMSE (log):", study.best_value)
print("Best params:", study.best_params)

[I 2026-05-28 16:00:15,554] A new study created in memory with name: no-name-1d95942f-2a62-4370-a92e-20d6925b70ef
C:\Users\Doruk\AppData\Local\Temp\ipykernel_16004\1512795407.py:11: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate": trial.suggest_loguniform("learning_rate", 1e-3, 0.3),
C:\Users\Doruk\AppData\Local\Temp\ipykernel_16004\1512795407.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "gamma": trial.suggest_loguniform("gamma", 1e-8, 10.0),
C:\Users\Doruk\AppData\Local\Temp\ipykernel_16004\1512795407.py:17: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna

Best RMSE (log): 0.12385364428026753
Best params: {'learning_rate': 0.0021670031745004196, 'max_depth': 3, 'min_child_weight': 2, 'subsample': 0.780961132983188, 'colsample_bytree': 0.5127372304098794, 'gamma': 0.0011043877275470951, 'reg_alpha': 6.576356509728523e-07, 'reg_lambda': 0.07652103702408468}


after searchin for the best hyperparameters, the prediction capabilites of the model has improved noticably

# xgboost regressor

Best RMSE (log): 0.12385364428026753
Best params: {'learning_rate': 0.0021670031745004196, 'max_depth': 3, 'min_child_weight': 2, 'subsample': 0.780961132983188, 'colsample_bytree': 0.5127372304098794, 'gamma': 0.0011043877275470951, 'reg_alpha': 6.576356509728523e-07, 'reg_lambda': 0.07652103702408468}

# pls regressor

Best RMSE (log): 0.1268944940593323
Best params: {'n_components': 19, 'scale': True}


# gp regressor

Best RMSE (log): 0.12449740338630581
Best params: {'length_scale': 2.94840898986105, 'noise_level': 0.025942747862994085, 'normalize_y': False}

In [38]:
# train the final models with the best hyperparameters, trained on the whole training data, and make predictions on the test data

xgb_best_params = {'learning_rate': 0.0021670031745004196, 'max_depth': 3, 'min_child_weight': 2, 'subsample': 0.780961132983188, 'colsample_bytree': 0.5127372304098794, 'gamma': 0.0011043877275470951, 'reg_alpha': 6.576356509728523e-07, 'reg_lambda': 0.07652103702408468}

xgb_best_model = XGBRegressor(**xgb_best_params)

xgb_best_model.fit(X_train_transformed_2, y_train)

xgb_best_predictions = xgb_best_model.predict(X_test_transformed_2)

# save the predictions to a csv file for submission
submission = pd.DataFrame({
    "Id": test_data_raw["Id"],
    "SalePrice": np.expm1(xgb_best_predictions)  # inverse of log1p transformation
})

submission.to_csv("submission_xgb.csv", index=False)


In [39]:
gp_best_params = {'length_scale': 2.94840898986105, 'noise_level': 0.025942747862994085, 'normalize_y': False}

kernel = ConstantKernel(1.0) * RBF(length_scale=gp_best_params['length_scale']) + WhiteKernel(noise_level=gp_best_params['noise_level'])

gp_best_model = GaussianProcessRegressor(kernel=kernel, normalize_y=gp_best_params['normalize_y'], random_state=42)
gp_best_model.fit(X_train_transformed_1.toarray(), y_train)

gp_best_predictions = gp_best_model.predict(X_test_transformed_1.toarray())

submission = pd.DataFrame({
    "Id": test_data_raw["Id"],
    "SalePrice": np.expm1(gp_best_predictions)  # inverse of log1p transformation
})

submission.to_csv("submission_gp.csv", index=False)

In [40]:
pls_best_params = {'n_components': 19, 'scale': True}

pls_best_model = PLSRegression(n_components=pls_best_params['n_components'], scale=pls_best_params['scale'])
pls_best_model.fit(X_train_transformed_1.toarray(), y_train)
pls_best_predictions = pls_best_model.predict(X_test_transformed_1.toarray()).ravel()

submission = pd.DataFrame({
    "Id": test_data_raw["Id"],
    "SalePrice": np.expm1(pls_best_predictions)  # inverse of log1p transformation
})
submission.to_csv("submission_pls.csv", index=False)